## An ANN Model
---

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
Dataset = pd.read_csv('assets/Churn_Modelling.csv')
df = Dataset
data = Dataset
data.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


> Removing Unnecessary Columns 
---

In [ ]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data.head(3)

> Encoding Categorical Column "Gender"
- Changing categorical into numerical value


In [ ]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

> OneHotEncoding to Geography
- Three values of Geo will be convert into 0, 1, 2. ANN considers them as greater and smaller. So OHE helps by over coming it.
---

In [ ]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()

geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder


In [ ]:
print(geo_encoder.toarray())

print(onehot_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

> Combining OHE with Original Dataset
---

In [ ]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head(3)

> Saving Encoder and Scaler
---

In [ ]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo, file)

> Dividing independent and dependent features
---

In [ ]:
X = data.drop('Exited', axis=1)
y = data['Exited']

> Train Test Split
---

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

>Standarization
---

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

> Saving Scaler file
---

In [ ]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

## ANN Implementations
> ANN Architecture

- <img src="assets/ann_arc.png">
> Workflow

- <img src="assets/workflow.png">


> Importing Libraries
---

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


> Builing ANN Model
---

In [ ]:
models = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  #HL1
    Dense(32, activation='relu'),                                   #HL2
    Dense(1, activation='sigmoid')                                  #O/p L
])

models.summary()

> Compile the model
- used for forward and backward propogation
---

In [ ]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

models.compile(optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])

> Setup TensorBoard
- Used for fit log visualization
---

In [ ]:
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

# log_dir = 'logs/fit' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = 'logs/fit/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

> Setup EarlyStopping
---

In [ ]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

> Train The Model
---

In [ ]:
history = models.fit(
    X_train, y_train, validation_data=(X_test,y_test), epochs=100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

> Saving Model
---

In [ ]:
models.save('models.h5')

> Loading TensorBoard
- Install extension: Tensorboard
- Double Run 2nd block code else use port: as shows, ex: http://localhost:6006/?darkMode=true#timeseries
---


In [ ]:
# %pip install tensorboard
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit/